# Task Timing Analysis — Gemiddeld over meerdere logs
Laadt alle CSV logs van een methode (bijv. `baseline_random_collab`),
berekent per taak waiting/processing/total time, en toont het **gemiddelde over alle episodes**.

In [1]:
# ── Configuration ──────────────────────────────────────────────────────
RUN_DIR = "../../results/runs/cvs/regular/20260301_130635/baseline_random_collab"

# Which logs to use: "final_eval", "eval", or "train"
LOG_TYPE = "train"

# Set to True to only count Mon-Fri 08:00-20:00 working hours
USE_WORKING_TIME = True

In [2]:
import sys, os, re, glob
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath("../../src"))

if USE_WORKING_TIME:
    from environment.work_schedule import working_seconds_between

# ── Timestamp parsing ─────────────────────────────────────────────────
def safe_parse_timestamps(series: pd.Series) -> pd.Series:
    def normalize(ts_str):
        if not isinstance(ts_str, str):
            return ts_str
        ts_str = re.sub(r'(\d{2}:\d{2}:\d{2}\.\d{6})\d+', r'\1', ts_str)
        ts_str = re.sub(r'(\d{2}:\d{2}:\d{2})([+-])', r'\1.000000\2', ts_str)
        return ts_str
    return pd.to_datetime(series.map(normalize), utc=True)

# ── Discover log files ────────────────────────────────────────────────
log_dir = os.path.join(RUN_DIR, "logs", LOG_TYPE)
if not os.path.isdir(log_dir):
    # fallback: final_eval -> eval
    if LOG_TYPE == "final_eval":
        log_dir = os.path.join(RUN_DIR, "logs", "eval")
        print(f"final_eval not found, falling back to eval/")
    if not os.path.isdir(log_dir):
        raise FileNotFoundError(f"Log directory not found: {log_dir}")

csv_files = sorted(glob.glob(os.path.join(log_dir, "*.csv")))
print(f"Found {len(csv_files)} log files in {log_dir}")
for f in csv_files:
    print(f"  - {os.path.basename(f)}")

Found 10 log files in ../../results/runs/cvs/regular/20260301_130635/baseline_random_collab/logs/train
  - log_train_ep0_20260301_130643.csv
  - log_train_ep1_20260301_131632.csv
  - log_train_ep2_20260301_132426.csv
  - log_train_ep3_20260301_133216.csv
  - log_train_ep4_20260301_133912.csv
  - log_train_ep5_20260301_134606.csv
  - log_train_ep6_20260301_135218.csv
  - log_train_ep7_20260301_140023.csv
  - log_train_ep8_20260301_141057.csv
  - log_train_ep9_20260301_141847.csv


In [3]:
# ── Load & compute timing for all logs ────────────────────────────────

def calc_seconds(row, start_col, end_col):
    start, end = row[start_col], row[end_col]
    if pd.isna(start) or pd.isna(end):
        return np.nan
    if USE_WORKING_TIME:
        return working_seconds_between(start, end)
    return (end - start).total_seconds()

all_task_stats = []   # per-task-type stats per episode
all_case_stats = []   # per-case stats per episode

for i, csv_file in enumerate(csv_files):
    df = pd.read_csv(csv_file)
    for col in ["task_assigned_time", "task_started_time", "task_completed_time"]:
        df[col] = safe_parse_timestamps(df[col])

    df["waiting_min"] = df.apply(calc_seconds, axis=1,
                                  start_col="task_assigned_time", end_col="task_started_time") / 60
    df["processing_min"] = df.apply(calc_seconds, axis=1,
                                     start_col="task_started_time", end_col="task_completed_time") / 60
    df["total_min"] = df.apply(calc_seconds, axis=1,
                                start_col="task_assigned_time", end_col="task_completed_time") / 60

    # Per task type
    task_grp = df.groupby("task_name")[["waiting_min", "processing_min", "total_min"]].mean()
    task_grp["episode"] = i
    all_task_stats.append(task_grp)

    # Per case
    case_grp = df.groupby("case_id").agg(
        n_tasks=("task_name", "count"),
        total_waiting_min=("waiting_min", "sum"),
        total_processing_min=("processing_min", "sum"),
        total_total_min=("total_min", "sum"),
        avg_waiting_min=("waiting_min", "mean"),
        avg_processing_min=("processing_min", "mean"),
    )
    # Case throughput
    case_times = df.groupby("case_id").agg(
        case_open=("task_assigned_time", "min"),
        case_close=("task_completed_time", "max"),
    )
    if USE_WORKING_TIME:
        case_grp["case_throughput_min"] = case_times.apply(
            lambda r: working_seconds_between(r["case_open"], r["case_close"]) / 60, axis=1
        )
    else:
        case_grp["case_throughput_min"] = (
            (case_times["case_close"] - case_times["case_open"]).dt.total_seconds() / 60
        )
    case_grp["episode"] = i
    all_case_stats.append(case_grp)

    print(f"  [{i+1}/{len(csv_files)}] {os.path.basename(csv_file)}: "
          f"{len(df)} tasks, {df['case_id'].nunique()} cases")

print(f"\nDone. Processed {len(csv_files)} episodes.")

  [1/10] log_train_ep0_20260301_130643.csv: 67963 tasks, 8192 cases
  [2/10] log_train_ep1_20260301_131632.csv: 67963 tasks, 8192 cases
  [3/10] log_train_ep2_20260301_132426.csv: 67963 tasks, 8192 cases
  [4/10] log_train_ep3_20260301_133216.csv: 67963 tasks, 8192 cases
  [5/10] log_train_ep4_20260301_133912.csv: 67963 tasks, 8192 cases
  [6/10] log_train_ep5_20260301_134606.csv: 67963 tasks, 8192 cases
  [7/10] log_train_ep6_20260301_135218.csv: 67963 tasks, 8192 cases
  [8/10] log_train_ep7_20260301_140023.csv: 67963 tasks, 8192 cases
  [9/10] log_train_ep8_20260301_141057.csv: 67963 tasks, 8192 cases
  [10/10] log_train_ep9_20260301_141847.csv: 67963 tasks, 8192 cases

Done. Processed 10 episodes.


In [4]:
# ── Gemiddelden per task type (over alle episodes) ────────────────────
task_all = pd.concat(all_task_stats)
task_avg = task_all.groupby("task_name")[["waiting_min", "processing_min", "total_min"]].agg(
    ["mean", "std"]
).round(2)

print("=" * 100)
print(f"GEMIDDELDEN PER TASK TYPE over {len(csv_files)} episodes (minutes)")
print(f"Working time: {'Mon-Fri 08:00-20:00' if USE_WORKING_TIME else 'Calendar (24/7)'}")
print("=" * 100)
display(task_avg)

GEMIDDELDEN PER TASK TYPE over 10 episodes (minutes)
Working time: Mon-Fri 08:00-20:00


waiting_min        processing_min        \
                                         mean    std           mean   std   
task_name                                                                   
Check DUR                                0.00   0.00           0.00  0.00   
Check Insurance                          0.00   0.00           0.00  0.00   
Check for Quality Assurance           6462.97  78.87           1.07  0.03   
Check if refill is allowed               0.00   0.00           0.00  0.00   
Contact Doctor about refill           6584.07  79.99          16.92  0.91   
Contact customer /  doctor            7171.72  82.76           8.20  1.05   
Enter prescription details            5433.86  76.18           2.69  0.04   
Pack the drugs (Production)           6600.82  81.03           3.20  0.05   
Pick-up                               5680.06  63.57           2.69  0.03   
Process drop-off                      4274.09  50.02           1.60  0.02   
Resolve DUR manually                  6638.64  80.58           1.85  0.06   
Resolve insurance issues manually     6594.10  81.24           4.07  0.16   

                                  total_min         
                                       mean    std  
task_name                                           
Check DUR                              0.00   0.00  
Check Insurance                        0.00   0.00  
Check for Quality Assurance         6464.04  78.86  
Check if refill is allowed             0.00   0.00  
Contact Doctor about refill         6600.99  80.71  
Contact customer /  doctor          7179.92  83.60  
Enter prescription details          5436.54  76.16  
Pack the drugs (Production)         6604.02  81.04  
Pick-up                             5682.75  63.58  
Process drop-off                    4275.69  50.02  
Resolve DUR manually                6640.49  80.60  
Resolve insurance issues manually   6598.17  81.29

In [5]:
# ── Gemiddelden per case (over alle episodes) ─────────────────────────
case_all = pd.concat(all_case_stats)
case_metrics = ["n_tasks", "total_waiting_min", "total_processing_min",
                "total_total_min", "avg_waiting_min", "avg_processing_min",
                "case_throughput_min"]

# Average per episode first, then average across episodes
episode_means = case_all.groupby("episode")[case_metrics].mean()
overall = episode_means.agg(["mean", "std"]).round(2)

print("=" * 100)
print(f"GEMIDDELDE CASE METRICS over {len(csv_files)} episodes (minutes)")
print("=" * 100)
display(overall)

# Per-episode breakdown
print("\n")
print("=" * 100)
print("PER EPISODE")
print("=" * 100)
display(episode_means.round(2))

GEMIDDELDE CASE METRICS over 10 episodes (minutes)


,n_tasks,total_waiting_min,total_processing_min,total_total_min,avg_waiting_min,avg_processing_min,case_throughput_min
mean,8.3,30441.50,13.01,30454.51,3651.13,1.55,30454.51
std,0.0,363.65,0.06,363.69,43.71,0.01,363.69




PER EPISODE


,n_tasks,total_waiting_min,total_processing_min,total_total_min,avg_waiting_min,avg_processing_min,case_throughput_min
episode,,,,,,,
0,8.3,30418.47,12.97,30431.44,3648.58,1.55,30431.44
1,8.3,30556.91,13.07,30569.98,3665.09,1.56,30569.98
2,8.3,30529.13,13.03,30542.16,3661.67,1.56,30542.16
3,8.3,29840.81,12.97,29853.78,3578.92,1.55,29853.78
4,8.3,30827.10,13.08,30840.18,3697.37,1.56,30840.18
5,8.3,30697.70,13.05,30710.75,3682.14,1.55,30710.75
6,8.3,30669.63,13.08,30682.71,3678.29,1.56,30682.71
7,8.3,29874.17,12.90,29887.07,3582.91,1.54,29887.07
8,8.3,30174.50,13.02,30187.52,3618.96,1.55,30187.52
